# R&D Space Market Counterfactual — Main Model

Pooled LightGBM dynamic panel model with 1-year lagged features, trained on 2006–2019 (pre-COVID) to predict counterfactual available R&D industrial space for 2020–2023 across 100 U.S. MSAs. Produces the `Structural_Gap` (actual minus counterfactual) that every downstream notebook in this repo consumes.

Includes: feature engineering with 1-year lags, MSA-level LOOCV validation, size-conditional bias correction, and SHAP global feature importance.

See `docs/methodology.md` for the full model specification and version history.

In [ ]:
""" FINAL MODEL — v14b (100-MSA panel: Hattiesburg/Gulfport-Biloxi excluded
    from the entire pipeline, not just presentation-facing steps)
    + true national LQ denominator + size-conditional bias correction
    + VMT-per-capita
R&D Space Market Counterfactual — Dynamic Panel (Lagged Features)
=================================================================
Pooled LightGBM with 1-year lagged features trained on 2006-2019
(pre-COVID), predicting counterfactual available space for 2020-2023.

Specification: log(Available_SF_Total_t) = f(RD_Econ_{t-1}, RE_{t-1}, Supply_{t-1}, BDS_{t-1}, VMT_{t-1}, Productivity_{t-1}, Patents_{t-1}, Year_FE)

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from scipy import stats
import shap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

# Presentation-set exclusions. As of v14b, this list is now applied
# IMMEDIATELY AFTER DATA CLEANING (Section 1B), removing both MSAs
# from the entire pipeline -- not just from presentation-facing steps
# as in v14 and earlier. Every filter below that still references
# NON_PRESENTATION_MSAS is now a no-op (kept for structural
# consistency with the v14 script, not because it does anything here).
NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']

# ══════════════════════════════════════════════════════════════════
# TRUE NATIONAL ADVANCED-INDUSTRY EMPLOYMENT SHARE (v14)
# ══════════════════════════════════════════════════════════════════
TRUE_NATIONAL_ADV_SHARE = {
    2005: 0.085155, 2006: 0.085383, 2007: 0.085491, 2008: 0.085975,
    2009: 0.083759, 2010: 0.083277, 2011: 0.084572, 2012: 0.085251,
    2013: 0.085194, 2014: 0.085128, 2015: 0.085325, 2016: 0.084759,
    2017: 0.084385, 2018: 0.085737, 2019: 0.086608, 2020: 0.090170,
    2021: 0.089849, 2022: 0.090781, 2023: 0.090544,
}

# ══════════════════════════════════════════════════════════════════
# 1. LOAD & CLEAN
# ══════════════════════════════════════════════════════════════════
file_path = "/kaggle/input/datasets/utkarshadahal/finaldataset/Panel_CoStar_Fused_Full_WITH_AVAIL_CONSTR_BDS_VMT_PATENTS.csv"
df = pd.read_csv(file_path)
print(f"Dataset loaded. Shape: {df.shape}")
print(f"MSAs: {df['MSA_Name'].nunique()} | Years: {sorted(df['Year'].unique())}")

df.columns = df.columns.str.strip()
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and col != 'MSA_Name':
        df[col] = df[col].astype(str).str.replace(r'[\$, ]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ══════════════════════════════════════════════════════════════════
# 1B. DATA CLEANING -- recode disguised-missing zeros to NaN
# ══════════════════════════════════════════════════════════════════
def recode_disguised_zeros(frame):
    frame = frame.copy()
    mask = frame['CoStar_Median_Cap_Rate'] == 0
    frame.loc[mask, 'CoStar_Median_Cap_Rate'] = np.nan
    n1 = mask.sum()
    pretrack_26 = frame['Year'].isin([2005, 2006])
    mask2 = (frame['Gross_Delivered_Buildings'] == 0) & pretrack_26
    frame.loc[mask2, 'Gross_Delivered_Buildings'] = np.nan
    n2 = mask2.sum()
    mask3 = (frame['Construction_Starts_SF'] == 0) & pretrack_26
    frame.loc[mask3, 'Construction_Starts_SF'] = np.nan
    n3 = mask3.sum()
    mask4 = (frame['CoStar_Sales_Transactions'] == 0) & pretrack_26
    frame.loc[mask4, 'CoStar_Sales_Transactions'] = np.nan
    n4 = mask4.sum()
    pretrack_27 = frame['Year'].isin([2005, 2006, 2007])
    mask5 = (frame['CoStar_Net_Absorption_SF'] == 0) & pretrack_27
    frame.loc[mask5, 'CoStar_Net_Absorption_SF'] = np.nan
    n5 = mask5.sum()
    print(f"Recoded disguised-missing zeros -> NaN:")
    print(f"  CoStar_Median_Cap_Rate   : {n1} rows (all years, 0% never real)")
    print(f"  Gross_Delivered_Buildings: {n2} rows (2005-2006 only)")
    print(f"  Construction_Starts_SF   : {n3} rows (2005-2006 only)")
    print(f"  CoStar_Sales_Transactions: {n4} rows (2005-2006 only)")
    print(f"  CoStar_Net_Absorption_SF : {n5} rows (2005-2007 only)")
    print(f"  Genuine zeros from later years are left untouched.")
    return frame

df = recode_disguised_zeros(df)

# ══════════════════════════════════════════════════════════════════
# 1C. (NEW v14b) EXCLUDE NON-PRESENTATION MSAs FROM THE ENTIRE PIPELINE
# ══════════════════════════════════════════════════════════════════
_n_rows_before_exclusion = len(df)
_n_msas_before_exclusion = df['MSA_Name'].nunique()
df = df[~df['MSA_Name'].isin(NON_PRESENTATION_MSAS)].reset_index(drop=True)
print(f"\n{'='*60}")
print(f"v14b -- EXCLUDING {NON_PRESENTATION_MSAS} FROM ENTIRE PIPELINE")
print(f"{'='*60}")
print(f"  Rows before exclusion: {_n_rows_before_exclusion} ({_n_msas_before_exclusion} MSAs)")
print(f"  Rows after exclusion : {len(df)} ({df['MSA_Name'].nunique()} MSAs)")
print(f"  Rows removed         : {_n_rows_before_exclusion - len(df)}")

# ══════════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING (contemporaneous)
# ══════════════════════════════════════════════════════════════════
for col in ['Total_GDP', 'All_Total_Employees', 'All_Total_Wages',
            'All_Total_Establishments', 'Adv_Ind_Employees',
            'Adv_Ind_Total_Wages_($)', 'Total_Population',
            'CoStar_Asset_Value', 'CoStar_Inventory_SF']:
    if col in df.columns:
        df[f'log_{col}'] = np.log(df[col].clip(lower=1))

for col in ['interstate vmt', 'local vmt', 'total vmt']:
    if col in df.columns:
        safe_col = col.replace(' ', '_')
        df[f'log_{safe_col}'] = np.log(df[col].clip(lower=1))

if 'patent_count' in df.columns:
    df['log_patent_count'] = np.log(df['patent_count'].clip(lower=1))

df['total_vmt_per_capita'] = df['total vmt'] / df['Total_Population'].clip(lower=1)
df['log_total_vmt_per_capita'] = np.log(df['total_vmt_per_capita'].clip(lower=1e-6))

df['RD_Intensity']         = df['Adv_Ind_GDP'] / df['Total_GDP'].clip(lower=1)
df['Adv_Ind_Emp_Share']    = (df['Adv_Ind_Employees'] /
                               df['All_Total_Employees'].clip(lower=1))
df['Adv_Ind_Wage_Premium'] = (df['Adv_Ind_Avg_Annual_Wages_per_Worker_($)'] /
                               df['All_Avg_Wage_Per_Worker'].clip(lower=1))
df['Productivity'] = df['Adv_Ind_GDP'] / df['Adv_Ind_Employees'].clip(lower=1)

if 'ZHVI' in df.columns:
    df['log_ZHVI'] = np.log(df['ZHVI'].clip(lower=1))
if 'ZORI' in df.columns:
    df['log_ZORI'] = np.log(df['ZORI'].clip(lower=1))

if 'CoStar_Net_Absorption_SF' in df.columns:
    df['Absorption_Rate'] = (df['CoStar_Net_Absorption_SF'] /
                              df['CoStar_Inventory_SF'].clip(lower=1))

df_sorted = df.sort_values(['MSA_Name', 'Year'])
df['RD_Intensity_5yr_CAGR'] = (
    df_sorted.groupby('MSA_Name')['RD_Intensity']
    .transform(lambda x: x / x.shift(5).clip(lower=0.001))
    .clip(lower=0) ** (1/5) - 1
)

df['Vacant_Space_SF'] = df['CoStar_Vacancy_Rate'] * df['CoStar_Inventory_SF']
df['log_Available_SF_Total'] = np.log(df['Available_SF_Total'].clip(lower=1))

_unmapped_years = sorted(set(df['Year'].unique()) - set(TRUE_NATIONAL_ADV_SHARE.keys()))
if _unmapped_years:
    raise SystemExit(
        f"STOPPING -- TRUE_NATIONAL_ADV_SHARE has no entry for year(s) {_unmapped_years}. "
        f"Extend the dictionary with true national values for these years before proceeding."
    )
natl_adv_share    = df['Year'].map(TRUE_NATIONAL_ADV_SHARE)
natl_rd_intensity = df.groupby('Year')['RD_Intensity'].transform('mean')
natl_rd_exp_avg = df.groupby('Year').apply(
    lambda x: (x['Total_R&D_Expenditure_(x1000)'] / x['Total_GDP'].clip(lower=1)).mean()
).reset_index(name='natl_rd_exp_share')
df = df.merge(natl_rd_exp_avg, on='Year', how='left')

df['LQ_AdvInd_Emp']    = df['Adv_Ind_Emp_Share'] / natl_adv_share.clip(lower=0.001)
df['LQ_RD_Intensity']  = df['RD_Intensity'] / natl_rd_intensity.clip(lower=0.001)
df['LQ_RD_Expenditure'] = (
    (df['Total_R&D_Expenditure_(x1000)'] / df['Total_GDP'].clip(lower=1)) /
    df['natl_rd_exp_share'].clip(lower=0.001))

print(f"\nv14 check -- LQ_AdvInd_Emp uses TRUE national denominator "
      f"(range {min(TRUE_NATIONAL_ADV_SHARE.values()):.4f}-"
      f"{max(TRUE_NATIONAL_ADV_SHARE.values()):.4f} across 2005-2023).")

df['Inventory_Growth'] = df.groupby('MSA_Name')['CoStar_Inventory_SF'].pct_change()
df['Space_Per_AdvInd_Worker']     = (df['CoStar_Inventory_SF'] / df['Adv_Ind_Employees'].clip(lower=1))
df['log_Space_Per_AdvInd_Worker'] = np.log(df['Space_Per_AdvInd_Worker'].clip(lower=0.1))
df['Rent_To_AdvInd_Wage'] = (
    df['CoStar_Rent_Overall'] / df['Adv_Ind_Avg_Annual_Wages_per_Worker_($)'].clip(lower=1) * 1000)

df['Delivery_Intensity'] = (df['Gross_Delivered_Buildings'] / df['CoStar_Inventory_SF'].clip(lower=1))
df['Avg_Delivered_Building_Size'] = np.where(
    df['Gross_Delivered_Buildings'] > 0,
    df['Construction_Starts_SF'] / df['Gross_Delivered_Buildings'].replace(0, np.nan),
    np.nan
)

_monthly_absorption = df['CoStar_Net_Absorption_SF'] / 12
_floor = 0.01 * df['CoStar_Inventory_SF'].clip(lower=1) / 12
_safe_denom = np.where(
    _monthly_absorption.abs() < _floor,
    np.sign(_monthly_absorption.replace(0, 1)) * _floor,
    _monthly_absorption
)
df['Months_of_Supply'] = (df['Available_SF_Total'] / _safe_denom).clip(-60, 60)

df['log_Adv_Ind_Establishments'] = np.log(df['Adv_Ind_Establishments'].clip(lower=1))
df['log_All_Avg_Wage_Per_Worker'] = np.log(df['All_Avg_Wage_Per_Worker'].clip(lower=1))
df['log_Adv_Ind_Avg_Annual_Wages_per_Worker'] = np.log(
    df['Adv_Ind_Avg_Annual_Wages_per_Worker_($)'].clip(lower=1))
df['log_median_income'] = np.log(df['median_income'].clip(lower=1))
df['log_poverty_count'] = np.log(df['poverty_count'].clip(lower=1))
df['log_CoStar_Rent_Overall'] = np.log(df['CoStar_Rent_Overall'].clip(lower=1))
df['log_CoStar_Sales_Transactions'] = np.log(df['CoStar_Sales_Transactions'].clip(lower=1))
df['log_Productivity'] = np.log(df['Productivity'].clip(lower=1))
df['log_Avg_Delivered_Building_Size'] = np.log(df['Avg_Delivered_Building_Size'].clip(lower=1))
df['log_Construction_Starts_SF_12Mo'] = np.log(df['Construction_Starts_SF_12Mo'].clip(lower=1))
df['log_Business_Entry_Rate'] = np.log(df['Business_Entry_Rate'].clip(lower=1))
df['log_Business_Exit_Rate'] = np.log(df['Business_Exit_Rate'].clip(lower=1))
df['log_RD_Intensity'] = np.log(df['RD_Intensity'].clip(lower=0.0001))
df['log_Adv_Ind_Emp_Share'] = np.log(df['Adv_Ind_Emp_Share'].clip(lower=0.0001))
df['log_CoStar_Cap_Rate'] = np.log(df['CoStar_Cap_Rate'].clip(lower=0.0001))
df['log_Rent_To_AdvInd_Wage'] = np.log(df['Rent_To_AdvInd_Wage'].clip(lower=0.0001))
df['log_Adv_Ind_Wage_Premium'] = np.log(df['Adv_Ind_Wage_Premium'].clip(lower=0.01))
df['log_Delivery_Intensity'] = np.log(df['Delivery_Intensity'].clip(lower=1e-10))

print("Features engineered.")

# ══════════════════════════════════════════════════════════════════
# 3. BUILD LAGGED FEATURES
# ══════════════════════════════════════════════════════════════════
BASE_FEATURES = [
    'log_Adv_Ind_Employees', 'log_Adv_Ind_Establishments', 'log_All_Avg_Wage_Per_Worker',
    'log_Adv_Ind_Avg_Annual_Wages_per_Worker', 'log_RD_Intensity', 'log_Adv_Ind_Emp_Share',
    'log_Adv_Ind_Wage_Premium', 'Number_of_Universities', 'Total_R&D_Expenditure_(x1000)',
    'Avg_R&D_per_Institution_(x1000)', 'log_Productivity', 'log_median_income',
    'log_poverty_count', 'edu_bachelor_plus_pct', 'CoStar_Vacancy_Rate', 'log_CoStar_Rent_Overall',
    'CoStar_Rent_Growth_12Mo', 'CoStar_Cap_Rate', 'CoStar_Market_Cap_Rate',
    'log_CoStar_Sales_Transactions', 'LQ_RD_Expenditure', 'log_Rent_To_AdvInd_Wage',
    'log_Business_Entry_Rate', 'log_Business_Exit_Rate', 'Mature_Firm_Share',
    'log_total_vmt_per_capita', 'log_patent_count',
    'log_Construction_Starts_SF_12Mo', 'log_Delivery_Intensity', 'log_Avg_Delivered_Building_Size',
    'Months_of_Supply',
]
BASE_FEATURES = [f for f in BASE_FEATURES if f in df.columns]

df['Year_FE'] = (df['Year'] - df['Year'].min()).astype(int)
df = df.sort_values(['MSA_Name', 'Year']).reset_index(drop=True)

FEATURE_LAG_OVERRIDES = {
    'log_total_vmt_per_capita': 0,
    'log_patent_count': 0,
    'log_Adv_Ind_Avg_Annual_Wages_per_Worker': 0,
    'log_median_income': 0,
}

for feat in BASE_FEATURES:
    lag = FEATURE_LAG_OVERRIDES.get(feat, 1)
    if lag == 0:
        df[f'{feat}_lag0'] = df[feat]
    else:
        df[f'{feat}_lag{lag}'] = df.groupby('MSA_Name')[feat].shift(lag)

FEATURES     = [f'{f}_lag{FEATURE_LAG_OVERRIDES.get(f, 1)}' for f in BASE_FEATURES] + ['Year_FE']
CAT_FEAT_IDX = [FEATURES.index('Year_FE')]

ZERO_INFLATED_LAG = [
    'Number_of_Universities_lag1',
    'Total_R&D_Expenditure_(x1000)_lag1',
    'Avg_R&D_per_Institution_(x1000)_lag1',
]

n_lag0 = sum(1 for f in BASE_FEATURES if FEATURE_LAG_OVERRIDES.get(f, 1) == 0)
print(f"\nFeature set: {len(FEATURES)} features "
      f"({len(BASE_FEATURES) - n_lag0} at lag-1, {n_lag0} at lag-0) + Year_FE")

# ══════════════════════════════════════════════════════════════════
# 4. DROP ROWS WITH NULL TARGET OR ALL-NULL LAGS
# ══════════════════════════════════════════════════════════════════
df = df[df['Available_SF_Total'].notna() & (df['Available_SF_Total'] > 0)].reset_index(drop=True)
lag_null_mask = df[FEATURES].isnull().all(axis=1)
df = df[~lag_null_mask].reset_index(drop=True)

print(f"Working dataset: {len(df)} rows | Years: {sorted(df['Year'].unique())} | "
      f"MSAs: {df['MSA_Name'].nunique()}")

# ══════════════════════════════════════════════════════════════════
# 5. TRAIN / PREDICT SPLIT
# ══════════════════════════════════════════════════════════════════
TRAIN_YEARS   = list(range(2006, 2020))
PREDICT_YEARS = [2020, 2021, 2022, 2023]

df_train   = df[df['Year'].isin(TRAIN_YEARS)].copy().reset_index(drop=True)
df_predict = df[df['Year'].isin(PREDICT_YEARS)].copy().reset_index(drop=True)

print(f"\n{'='*60}\nTRAIN / PREDICT SPLIT (v14b -- 100-MSA panel throughout)\n{'='*60}")
print(f"  Train   (2006-2019): {len(df_train)} rows, {df_train['MSA_Name'].nunique()} MSAs")
print(f"  Predict (2020-2023): {len(df_predict)} rows, {df_predict['MSA_Name'].nunique()} MSAs")

# ══════════════════════════════════════════════════════════════════
# 6. KNN IMPUTATION -- fit on training set only
# ══════════════════════════════════════════════════════════════════
impute_feats = [f for f in FEATURES if f not in ZERO_INFLATED_LAG and f != 'Year_FE']

print(f"\nMissing values in training set before imputation:")
missing_tr = df_train[FEATURES].isnull().sum()
print(missing_tr[missing_tr > 0].to_string())

knn = KNNImputer(n_neighbors=5)

X_tr_imp = pd.DataFrame(
    knn.fit_transform(df_train[impute_feats].replace([np.inf, -np.inf], np.nan)),
    columns=impute_feats)
X_train = pd.concat([
    X_tr_imp,
    df_train[ZERO_INFLATED_LAG].reset_index(drop=True),
    df_train[['Year_FE']].reset_index(drop=True),
], axis=1)[FEATURES]
y_train = df_train['log_Available_SF_Total'].values

X_pr_imp = pd.DataFrame(
    knn.transform(df_predict[impute_feats].replace([np.inf, -np.inf], np.nan)),
    columns=impute_feats)
X_predict = pd.concat([
    X_pr_imp,
    df_predict[ZERO_INFLATED_LAG].reset_index(drop=True),
    df_predict[['Year_FE']].reset_index(drop=True),
], axis=1)[FEATURES]
y_predict = df_predict['log_Available_SF_Total'].values

print(f"\nAfter imputation:")
print(f"  X_train nulls  : {X_train.isnull().sum().sum()}")
print(f"  X_predict nulls: {X_predict.isnull().sum().sum()}")
print(f"  X_train shape  : {X_train.shape}")
print(f"  X_predict shape: {X_predict.shape}")

# ══════════════════════════════════════════════════════════════════
# 7. LIGHTGBM MODEL
# ══════════════════════════════════════════════════════════════════
def make_lgb(seed=42, n_est=200):
    return LGBMRegressor(
        n_estimators=n_est, learning_rate=0.05, max_depth=4, num_leaves=10,
        min_child_samples=40, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=3.0, random_state=seed, verbose=-1)

lgb_model = make_lgb(n_est=300)
lgb_model.fit(X_train.values, y_train, categorical_feature=CAT_FEAT_IDX)

y_pred_train   = lgb_model.predict(X_train.values)
r2_insample    = r2_score(y_train, y_pred_train)
mae_insample   = mean_absolute_error(y_train, y_pred_train)
mdape_insample = np.median(np.abs((y_train - y_pred_train) / np.clip(y_train, 1e-6, None))) * 100

tr_2018 = df_train[df_train['Year'] <= 2018]
te_2019 = df_train[df_train['Year'] == 2019]

knn_h = KNNImputer(n_neighbors=5)
X_h_tr_imp = pd.DataFrame(
    knn_h.fit_transform(tr_2018[impute_feats].replace([np.inf, -np.inf], np.nan)),
    columns=impute_feats)
X_h_tr = pd.concat([
    X_h_tr_imp, tr_2018[ZERO_INFLATED_LAG].reset_index(drop=True),
    tr_2018[['Year_FE']].reset_index(drop=True),
], axis=1)[FEATURES]
y_h_tr = tr_2018['log_Available_SF_Total'].values

X_h_te_imp = pd.DataFrame(
    knn_h.transform(te_2019[impute_feats].replace([np.inf, -np.inf], np.nan)),
    columns=impute_feats)
X_h_te = pd.concat([
    X_h_te_imp, te_2019[ZERO_INFLATED_LAG].reset_index(drop=True),
    te_2019[['Year_FE']].reset_index(drop=True),
], axis=1)[FEATURES]
y_h_te = te_2019['log_Available_SF_Total'].values

val_model = make_lgb(n_est=300)
val_model.fit(X_h_tr.values, y_h_tr, categorical_feature=CAT_FEAT_IDX)
y_pred_2019 = val_model.predict(X_h_te.values)
r2_2019     = r2_score(y_h_te, y_pred_2019)
mae_2019    = mean_absolute_error(y_h_te, y_pred_2019)
mdape_2019  = np.median(np.abs((y_h_te - y_pred_2019) / np.clip(y_h_te, 1e-6, None))) * 100

print("\nRunning MSA-level LOOCV (2006-2019 only)...")
msa_list     = df_train['MSA_Name'].unique()
loo_preds_tr = np.full(len(y_train), np.nan)

for i, msa in enumerate(msa_list):
    test_idx  = df_train[df_train['MSA_Name'] == msa].index.tolist()
    train_idx = df_train[df_train['MSA_Name'] != msa].index.tolist()
    if not train_idx or not test_idx:
        continue
    m_ = make_lgb(n_est=200)
    m_.fit(X_train.values[train_idx], y_train[train_idx], categorical_feature=CAT_FEAT_IDX)
    loo_preds_tr[test_idx] = m_.predict(X_train.values[test_idx])
    if (i + 1) % 20 == 0:
        print(f"  LOOCV: {i+1}/{len(msa_list)} MSAs done...")

valid_mask = ~np.isnan(loo_preds_tr)
loo_r2     = r2_score(y_train[valid_mask], loo_preds_tr[valid_mask])
loo_mae    = mean_absolute_error(y_train[valid_mask], loo_preds_tr[valid_mask])
loo_mdape  = np.median(np.abs(
    (y_train[valid_mask] - loo_preds_tr[valid_mask]) / np.clip(y_train[valid_mask], 1e-6, None))) * 100

df_loocv = df_train[['MSA_Name', 'Year', 'log_Available_SF_Total', 'Available_SF_Total']].copy()
df_loocv['LOOCV_Counterfactual'] = loo_preds_tr
df_loocv['LOOCV_Gap']            = df_loocv['log_Available_SF_Total'] - df_loocv['LOOCV_Counterfactual']
df_loocv['Period'] = 'Pre-COVID (2006-2019)'
df_loocv.to_csv("AvailSFTotal_LOOCV_Residuals.csv", index=False)
print(f"\nLOOCV residuals exported: {len(df_loocv)} rows")
print(f"  LOOCV mean gap: {df_loocv['LOOCV_Gap'].mean():+.4f} log units")
print(f"  LOOCV gap std:  {df_loocv['LOOCV_Gap'].std():.4f} log units")

r2_seeds = []
for s in [0, 42, 99]:
    m_ = make_lgb(seed=s, n_est=300)
    m_.fit(X_train.values, y_train, categorical_feature=CAT_FEAT_IDX)
    r2_seeds.append(r2_score(y_train, m_.predict(X_train.values)))

print(f"\n{'='*70}")
print("AVAILABLE SF TOTAL DYNAMIC PANEL -- TRAINED ON 2006-2019 (v14b: 100-MSA panel + true national LQ + size-conditional bias correction + VMT-per-capita)")
print(f"Specification: log(Available_SF_Total_t) = f(RD_Econ_{{t-1}}, RE_{{t-1}}, Supply_{{t-1}}, BDS_{{t-1}}, VMT_{{t-1}}, Productivity_{{t-1}}, Patents_{{t-1}}, Year_FE)")
print(f"{'='*70}")
print(f"  Train obs (2006-2019)  : {len(y_train)}")
print(f"  Predict obs (2020-2023): {len(y_predict)}")
print(f"  MSAs                   : {df_train['MSA_Name'].nunique()}")
print(f"  Features               : {len(FEATURES)}")
print(f"  R^2 in-sample          : {r2_insample:.4f}")
print(f"  In-sample MAE          : {mae_insample:.4f} (log units)")
print(f"  In-sample MdAPE        : {mdape_insample:.1f}%")
print(f"  2019 holdout R^2       : {r2_2019:.4f}")
print(f"  2019 holdout MAE       : {mae_2019:.4f} (log units)")
print(f"  2019 holdout MdAPE     : {mdape_2019:.1f}%")
print(f"  MSA LOOCV R^2          : {loo_r2:.4f}")
print(f"  MSA LOOCV MAE          : {loo_mae:.4f} (log units)")
print(f"  MSA LOOCV MdAPE        : {loo_mdape:.1f}%")
print(f"  Stability std          : {np.std(r2_seeds):.5f} "
      f"({'OK' if np.std(r2_seeds) < 0.01 else 'UNSTABLE'})")
print(f"  R^2 across seeds       : {[round(r,4) for r in r2_seeds]}")
print(f"  NOTE (v14b): these numbers are from a model TRAINED on 100 MSAs")
print(f"        (Hattiesburg/Gulfport-Biloxi excluded from the entire pipeline),")
print(f"        NOT the v14 102-MSA-trained model. Compare against the v14")
print(f"        console output before assuming these are the same.")

# ══════════════════════════════════════════════════════════════════
# 8. COUNTERFACTUAL PREDICTION 2020-2023 (pre-correction)
# ══════════════════════════════════════════════════════════════════
df_train['Counterfactual_LogSpace'] = y_pred_train
df_train['Counterfactual_Space_SF'] = np.exp(y_pred_train)
df_train['Structural_Gap']          = df_train['log_Available_SF_Total'] - df_train['Counterfactual_LogSpace']
df_train['Structural_Gap_SF']       = df_train['Available_SF_Total'] - df_train['Counterfactual_Space_SF']
df_train['Period'] = 'Pre-COVID (2006-2019)'

df_predict['Counterfactual_LogSpace'] = lgb_model.predict(X_predict.values)
df_predict['Counterfactual_Space_SF'] = np.exp(df_predict['Counterfactual_LogSpace'])
df_predict['Structural_Gap']    = df_predict['log_Available_SF_Total'] - df_predict['Counterfactual_LogSpace']
df_predict['Structural_Gap_SF'] = df_predict['Available_SF_Total'] - df_predict['Counterfactual_Space_SF']
df_predict['Period'] = df_predict['Year'].map({
    2020: 'COVID (2020)', 2021: 'COVID (2021)',
    2022: 'Recovery (2022)', 2023: 'Recovery (2023)'})

df_train['Structural_Gap_RAW']    = df_train['Structural_Gap']
df_train['Structural_Gap_SF_RAW'] = df_train['Structural_Gap_SF']
df_predict['Structural_Gap_RAW']    = df_predict['Structural_Gap']
df_predict['Structural_Gap_SF_RAW'] = df_predict['Structural_Gap_SF']

def classify_gap(gap, sigma):
    z = gap / sigma
    if   z >=  1.5: return "Significant Structural Surplus"
    elif z >=  0.5: return "Moderate Structural Surplus"
    elif z >  -0.5: return "Structurally Balanced"
    elif z >  -1.5: return "Moderate Structural Deficit"
    else:           return "Significant Structural Deficit"

LOOCV_SIGMA_RAW = df_loocv['LOOCV_Gap'].std()
print(f"\nLOOCV noise floor (sigma), BEFORE size correction: {LOOCV_SIGMA_RAW:.4f} log units")

# ══════════════════════════════════════════════════════════════════
# 8A. SIZE-CONDITIONAL BIAS CORRECTION
#     NOTE (v14b): size_bias_fit_data's "~NON_PRESENTATION_MSAS" filter
#     is now a no-op -- both MSAs are already absent from df_loocv.
#     This line is left in place, unchanged, for structural consistency
#     with the v14 script.
# ══════════════════════════════════════════════════════════════════
size_bias_fit_data = df_loocv[~df_loocv['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
size_bias_slope, size_bias_intercept, size_bias_r, size_bias_p, size_bias_stderr = stats.linregress(
    np.log(size_bias_fit_data['Available_SF_Total'].clip(lower=1)), size_bias_fit_data['LOOCV_Gap'])

print(f"\n{'='*70}")
print(f"SIZE-BIAS REGRESSION (fit on {len(size_bias_fit_data)} training LOOCV_Gap rows -- "
      f"100-MSA panel, v14b)")
print(f"{'='*70}")
print(f"  LOOCV_Gap = {size_bias_intercept:+.4f} + {size_bias_slope:+.4f} * log(Available_SF_Total)")
print(f"  r={size_bias_r:+.4f}, p={size_bias_p:.4f}, std_err={size_bias_stderr:.4f}")

def predicted_size_bias(log_size):
    return size_bias_intercept + size_bias_slope * log_size

df_train['log_Size_forbias']   = np.log(df_train['Available_SF_Total'].clip(lower=1))
df_predict['log_Size_forbias'] = np.log(df_predict['Available_SF_Total'].clip(lower=1))

df_predict['Counterfactual_LogSpace'] = (
    df_predict['Counterfactual_LogSpace'] + predicted_size_bias(df_predict['log_Size_forbias']))
df_predict['Counterfactual_Space_SF'] = np.exp(df_predict['Counterfactual_LogSpace'])
df_predict['Structural_Gap']    = df_predict['log_Available_SF_Total'] - df_predict['Counterfactual_LogSpace']
df_predict['Structural_Gap_SF'] = df_predict['Available_SF_Total'] - df_predict['Counterfactual_Space_SF']

df_full = pd.concat([df_train, df_predict], ignore_index=True)

_size_bias_resid = size_bias_fit_data['LOOCV_Gap'] - predicted_size_bias(
    np.log(size_bias_fit_data['Available_SF_Total'].clip(lower=1)))
LOOCV_SIGMA = _size_bias_resid.std()
print(f"\n  LOOCV_SIGMA before size correction: {LOOCV_SIGMA_RAW:.4f}")
print(f"  LOOCV_SIGMA after size correction  : {LOOCV_SIGMA:.4f}  (used for classification from here on)")
print(f"  Thresholds: Significant = |z| >= 1.5 ({1.5*LOOCV_SIGMA:+.4f} log units)")
print(f"              Moderate    = |z| >= 0.5 ({0.5*LOOCV_SIGMA:+.4f} log units)")

df_full['Market_Category']     = df_full['Structural_Gap'].apply(lambda g: classify_gap(g, LOOCV_SIGMA))
df_full['Market_Category_RAW'] = df_full['Structural_Gap_RAW'].apply(lambda g: classify_gap(g, LOOCV_SIGMA_RAW))

_verify = df_full[
    df_full['Year'].isin(PREDICT_YEARS) &
    ~df_full['MSA_Name'].isin(NON_PRESENTATION_MSAS)
]
_r_before, _p_before = stats.pearsonr(
    np.log(_verify['Available_SF_Total'].clip(lower=1)), _verify['Structural_Gap_RAW'])
_r_after, _p_after = stats.pearsonr(
    np.log(_verify['Available_SF_Total'].clip(lower=1)), _verify['Structural_Gap'])

print(f"\n{'='*70}")
print("SIZE-BIAS CORRECTION VERIFICATION -- 2020-2023 predict period (100-MSA panel)")
print(f"{'='*70}")
print(f"  BEFORE correction (Structural_Gap_RAW): r={_r_before:+.4f} (p={_p_before:.4f})")
print(f"  AFTER  correction (Structural_Gap)     : r={_r_after:+.4f} (p={_p_after:.4f})")
if abs(_r_after) < 0.1:
    print(f"  --> Correction working as intended: size correlation is negligible.")
else:
    print(f"  --> WARNING: size correlation still present after correction -- investigate")
    print(f"      before trusting downstream classifications.")

_n_changed = (
    df_full[df_full['Year'].isin(PREDICT_YEARS) & ~df_full['MSA_Name'].isin(NON_PRESENTATION_MSAS)]
    .groupby('MSA_Name')
    .apply(lambda g: g['Market_Category'].mode().iat[0] != g['Market_Category_RAW'].mode().iat[0])
    .sum()
)
print(f"  {_n_changed} of {_verify['MSA_Name'].nunique()} MSAs changed "
      f"typical Market_Category vs. the uncorrected (RAW) classification.")

pd.DataFrame({
    'Metric': ['Size_Bias_Slope', 'Size_Bias_Intercept', 'Size_Bias_r', 'Size_Bias_p',
               'LOOCV_SIGMA_RAW', 'LOOCV_SIGMA_Corrected',
               'Predict_Period_r_Before', 'Predict_Period_r_After', 'N_MSAs_Category_Changed'],
    'Value': [size_bias_slope, size_bias_intercept, size_bias_r, size_bias_p,
              LOOCV_SIGMA_RAW, LOOCV_SIGMA, _r_before, _r_after, _n_changed],
}).to_csv("SizeBias_Correction_Verification.csv", index=False)
print(f"\nSaved: SizeBias_Correction_Verification.csv")

# ── 2020-2023 summary (size-corrected) ─────────────────────────────
df_covid = df_full[df_full['Year'].isin(PREDICT_YEARS)].copy()

print(f"\n{'='*70}")
print("AVAILABLE SF TOTAL STRUCTURAL BREAK -- 2020-2023 COUNTERFACTUAL  [SIZE-CORRECTED, 100-MSA PANEL]")
print(f"{'='*70}")
print(f"\n*** THIS IS THE TABLE-2-STYLE CLASSIFICATION DISTRIBUTION, computed")
print(f"*** directly on the 100-MSA panel -- resolves the 398/399/400 ambiguity. ***")
print(f"\nCategory Distribution (2020-2023 combined), {len(df_covid)} MSA-year observations:")
print(df_covid['Market_Category'].value_counts().to_string())
_cat_pct = (df_covid['Market_Category'].value_counts(normalize=True) * 100).round(1)
print(f"\nAs percentages:")
print(_cat_pct.to_string())

covid_avg = (df_covid.groupby('MSA_Name')['Structural_Gap']
             .mean().sort_values(ascending=False).reset_index())
covid_avg.columns = ['MSA_Name', 'Avg_Gap_2020_2023']

print(f"\nTop 15 Structural Surplus [SIZE-CORRECTED, 100-MSA PANEL]:")
print(covid_avg.head(15).to_string(index=False))
print(f"\nTop 12 Structural Deficit [SIZE-CORRECTED, 100-MSA PANEL]:")
print(covid_avg.tail(12).sort_values('Avg_Gap_2020_2023').to_string(index=False))

print(f"\n{'='*60}\nYEAR-BY-YEAR STRUCTURAL BREAK  [SIZE-CORRECTED, 100-MSA PANEL]\n{'='*60}")
yearly = df_full.groupby('Year').agg(
    Mean_Gap=('Structural_Gap', 'mean'),
    Median_Gap=('Structural_Gap', 'median'),
    Pct_Deficit=('Structural_Gap', lambda x: (x < -0.20).mean() * 100),
    Pct_Surplus=('Structural_Gap', lambda x: (x > 0.20).mean() * 100),
    Mean_Actual_LogSpace=('log_Available_SF_Total', 'mean'),
    Mean_Counterfactual_LogSpace=('Counterfactual_LogSpace', 'mean'),
    Mean_Actual_Space_SF=('Available_SF_Total', 'mean'),
    Mean_Counterfactual_Space_SF=('Counterfactual_Space_SF', 'mean'),
).round(4)
print(yearly.to_string())

# ══════════════════════════════════════════════════════════════════
# 8B/8C/9/10/11 — ADV-WEIGHTED SECTIONS, SHAP, FIGURES, EXPORT
#     Mechanism unchanged from v14; all NON_PRESENTATION_MSAS filters
#     below are no-ops under v14b (both MSAs already absent from
#     df_full). Kept for structural consistency with v14.
# ══════════════════════════════════════════════════════════════════
anchor_weights = (
    df_full[df_full['Year'].between(2015, 2018)]
    .groupby('MSA_Name')['LQ_AdvInd_Emp']
    .mean().rename('LQ_AdvInd_Emp_2015_2018').reset_index()
)
df_full = df_full.merge(anchor_weights, on='MSA_Name', how='left')

missing_weight = df_full[df_full['LQ_AdvInd_Emp_2015_2018'].isna()]['MSA_Name'].unique()
if len(missing_weight) > 0:
    print(f"WARNING: No 2015-2018 LQ_AdvInd_Emp weight for: {list(missing_weight)}")
    fallback = df_full.groupby('MSA_Name')['LQ_AdvInd_Emp'].mean().rename('LQ_Fallback').reset_index()
    df_full = df_full.merge(fallback, on='MSA_Name', how='left')
    df_full['LQ_AdvInd_Emp_2015_2018'] = df_full['LQ_AdvInd_Emp_2015_2018'].fillna(df_full['LQ_Fallback'])
    df_full = df_full.drop(columns=['LQ_Fallback'])

df_full['Adv_Weight_LQ'] = df_full['LQ_AdvInd_Emp_2015_2018'] / (1 + df_full['LQ_AdvInd_Emp_2015_2018'])
df_full['Adv_Weighted_Gap_SF'] = df_full['Structural_Gap_SF'] * df_full['Adv_Weight_LQ']

df_covid = df_full[df_full['Year'].isin(PREDICT_YEARS)].copy()
covid_avg_weighted = (
    df_covid.groupby('MSA_Name')['Adv_Weighted_Gap_SF'].mean().sort_values(ascending=False).reset_index())
covid_avg_weighted.columns = ['MSA_Name', 'Avg_Adv_Weighted_Gap_2020_2023']

print(f"\n{'='*70}")
print("ADVANCED-INDUSTRY-WEIGHTED STRUCTURAL GAP -- 2020-2023  [ADV-WEIGHTED, SIZE-CORRECTED, TRUE-NATIONAL-LQ, 100-MSA PANEL]")
print(f"{'='*70}")
print(f"\nTop 15:")
print(covid_avg_weighted.head(15).to_string(index=False))
print(f"\nBottom 12:")
print(covid_avg_weighted.tail(12).sort_values('Avg_Adv_Weighted_Gap_2020_2023').to_string(index=False))

covid_avg_weighted.to_csv("AvailSFTotal_COVID_AdvWeighted_Gap.csv", index=False)
print(f"\nSaved: AvailSFTotal_COVID_AdvWeighted_Gap.csv")

df_full['Adv_Weighted_Available_SF_Total'] = df_full['Available_SF_Total'] * df_full['Adv_Weight_LQ']
df_full['Adv_Weighted_Counterfactual_SF']  = df_full['Counterfactual_Space_SF'] * df_full['Adv_Weight_LQ']

nat_actual_weighted = df_full.groupby('Year').agg(
    Adv_Weighted_Actual=('Adv_Weighted_Available_SF_Total', 'mean'),
    Adv_Weighted_Counter=('Adv_Weighted_Counterfactual_SF', 'mean'),
).reset_index()
nat_actual_weighted['Adv_Weighted_Gap'] = (
    nat_actual_weighted['Adv_Weighted_Actual'] - nat_actual_weighted['Adv_Weighted_Counter'])
nat_actual_weighted.to_csv("AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_National.csv", index=False)
print(f"Saved: AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_National.csv")

df_full[[
    'MSA_Name', 'Year', 'Available_SF_Total', 'Counterfactual_Space_SF',
    'Adv_Weight_LQ', 'Adv_Weighted_Available_SF_Total', 'Adv_Weighted_Counterfactual_SF',
]].sort_values(['MSA_Name', 'Year']).to_csv(
    "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv", index=False)
print(f"Saved: AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv")

print("\nComputing SHAP values (2006-2019 training set)...")
explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_train.values)

def _display_name(f):
    if f.endswith('_lag0'):
        return f[:-5] + ' (t)'
    elif f.endswith('_lag1'):
        return f[:-5] + ' (t-1)'
    return f

display_names = [_display_name(f) for f in FEATURES]
shap_df = pd.DataFrame({
    'Feature':   display_names,
    'Mean_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

print(f"\nSHAP Feature Importance (2006-2019, 100-MSA panel):")
print(shap_df.to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 10. SHAP FIGURES (restored -- dropped from the condensed v14b
#     script; needs shap_values / X_train in memory, so this can't be
#     rebuilt as a separate post-hoc script the way the rank-
#     comparison table could)
# ══════════════════════════════════════════════════════════════════
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, pd.DataFrame(X_train.values, columns=display_names),
                   plot_type='bar', max_display=20, show=False)
plt.title(f"SHAP Feature Importance -- Dynamic Panel (2006-2019, v14b) [100-MSA PANEL]\n"
          f"All features lagged 1 year | AR(1) term excluded", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("lag_counterfactual_shap_bar_v14b.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: lag_counterfactual_shap_bar_v14b.png")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, pd.DataFrame(X_train.values, columns=display_names),
                   max_display=20, show=False)
plt.title("SHAP Beeswarm -- Direction & Magnitude (2006-2019, v14b) [100-MSA PANEL]",
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("lag_counterfactual_shap_beeswarm_v14b.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: lag_counterfactual_shap_beeswarm_v14b.png")

# ══════════════════════════════════════════════════════════════════
# 11. EXPORT
# ══════════════════════════════════════════════════════════════════
df_output = df_full[[
    'MSA_Name', 'Year', 'Period',
    'Available_SF_Total', 'Counterfactual_Space_SF',
    'Structural_Gap', 'Structural_Gap_SF', 'Market_Category',
    'Structural_Gap_RAW', 'Structural_Gap_SF_RAW', 'Market_Category_RAW',
    'RD_Intensity', 'Adv_Ind_Emp_Share',
    'LQ_AdvInd_Emp', 'LQ_RD_Intensity', 'Productivity',
    'Business_Entry_Rate', 'Business_Exit_Rate',
    'Young_Firm_Share', 'Mature_Firm_Share',
    'interstate vmt', 'local vmt', 'total vmt',
    'patent_count',
]].sort_values(['MSA_Name', 'Year']).reset_index(drop=True)

df_output.to_csv("AvailSFTotal_Counterfactual_Results.csv", index=False)
covid_avg.to_csv("AvailSFTotal_COVID_Avg_Gap.csv", index=False)
yearly.to_csv("AvailSFTotal_Yearly_Gap_Trend.csv")
shap_df.to_csv("AvailSFTotal_SHAP_importance.csv", index=False)
df_loocv.to_csv("AvailSFTotal_LOOCV_Residuals.csv", index=False)

pd.DataFrame({
    'Metric': [
        'Target', 'Train years', 'Predict years',
        'Train obs', 'Predict obs', 'MSAs', 'Features',
        'R^2 in-sample', 'In-sample MAE', 'In-sample MdAPE',
        '2019 holdout R^2', '2019 holdout MAE', '2019 holdout MdAPE',
        'MSA LOOCV R^2', 'MSA LOOCV MAE', 'MSA LOOCV MdAPE',
        'Stability std', 'R^2 across seeds',
    ],
    'Value': [
        'log_Available_SF_Total (CoStar-reported, direct)',
        '2006-2019', '2020-2023',
        len(y_train), len(y_predict),
        df_train['MSA_Name'].nunique(),
        f"{len(FEATURES)} (all lagged 1yr, AR term excluded)",
        r2_insample, mae_insample, mdape_insample,
        r2_2019, mae_2019, mdape_2019,
        loo_r2, loo_mae, loo_mdape,
        np.std(r2_seeds),
        str([round(r, 4) for r in r2_seeds]),
    ]
}).to_csv("Lag_Space_Model_Summary.csv", index=False)

print(f"\n{'='*60}")
print("DONE -- Dynamic Panel Available SF Total Counterfactual (v14b: 100-MSA panel throughout)")
print(f"{'='*60}")
print(f"  Train obs (2006-2019)  : {len(y_train)}")
print(f"  Predict obs (2020-2023): {len(y_predict)}  <-- USE THIS NUMBER, not 398/399")
print(f"  MSAs                   : {df_train['MSA_Name'].nunique()}")